# 3x Steane Code CNOT Logical Error Rates

This notebook benchmarks the logical CNOT experiment using three [[7,1,3]] Steane codes (one for control, one for target, one as ancilla) with ancilla-mediated homological-measurement construction. Visualization is intentionally skipped; only logical-rate data is saved.

In [ ]:
import importlib
import numpy as np
import sinter
import stim
from ldpc import SinterBpOsdDecoder

from pathlib import Path

from epic import (
    AllocCode,
    FreeCode,
    InitCode,
    PauliChar,
    PauliEigenState,
    PauliString,
    QECCompiler,
    ReadoutCode,
    StimLikeNoiseModel,
)
from epic.modules.qec_gadgets.transversal_gates.transversal_cnot import TransversalCNOT
from epic.modules.stabilizers_codes.css_code import CSSCode
from epic.modules.stabilizers_codes.rotated_surface_code import RotatedSurfaceCode

base_dir = Path("..").resolve()
logical_rates_dir = base_dir / "logical_rates"
logical_rates_dir.mkdir(parents=True, exist_ok=True)

In [6]:
# Define the [[7,1,3]] Steane code
# Parity check matrix for Steane code
H_steane = np.array(
    [
        [0, 0, 0, 1, 1, 1, 1],
        [0, 1, 1, 0, 0, 1, 1],
        [1, 0, 1, 0, 1, 0, 1],
    ],
    dtype=np.uint8,
)

# Logical operators for Steane code
lX = [1, 1, 1, 0, 0, 0, 0]
lZ = [1, 1, 1, 0, 0, 0, 0]

# Create three Steane codes: control, target, and ancilla
control_code = CSSCode.from_css_pcm(
    code_name="steane_ctrl",
    hx=H_steane,
    hz=H_steane,
    logical_qubits=[(lX, lZ)],
)

target_code = CSSCode.from_css_pcm(
    code_name="steane_target",
    hx=H_steane,
    hz=H_steane,
    logical_qubits=[(lX, lZ)],
)

ancilla_code = RotatedSurfaceCode.from_distance(distance=3, code_name="ancilla_d3_steane", system_coordinate=(1, 0))

print(f"Control code: {control_code.name}, n={control_code.n}, k={control_code.k}")
print(f"Target code: {target_code.name}, n={target_code.n}, k={target_code.k}")
print(f"Ancilla code: {ancilla_code.name}, n={ancilla_code.n}, k={ancilla_code.k}")

Control code: steane_ctrl, n=7, k=1
Target code: steane_target, n=7, k=1
Ancilla code: ancilla_d3_steane, n=9, k=1


In [7]:
PRIMITIVES_CONFIG = {
    "epic.core.qec_primitives.interfaces.ApplyGate": "epic.modules.qec_primitives.apply_gates.simple_gate_application.SimpleGateApplication",
    "epic.core.qec_primitives.interfaces.Readout": "epic.modules.qec_primitives.readouts.naive_readout.NaiveReadout",
    "epic.core.qec_primitives.interfaces.ExtractSyndrome": "epic.modules.qec_primitives.syndrome_extraction.zxcoloring_extraction.ZXColoringExtraction",
}


def compile_steane_transversal_cnot_test():
    """Compile minimal 2x Steane Z+ -> transversal CNOT -> Z readout experiment."""
    import epic.modules.qec_gadgets.transversal_gates.transversal_cnot as tcnot_module
    importlib.reload(tcnot_module)
    TransversalCNOT = tcnot_module.TransversalCNOT

    program = [
        AllocCode(target_code=control_code, code_varname="ctrl_patch", logical_qubits_varnames=["ctrl_lq_0"]),
        AllocCode(target_code=target_code, code_varname="target_patch", logical_qubits_varnames=["target_lq_0"]),
        InitCode(targets=["ctrl_patch"], initial_state=PauliEigenState.Z_plus, tag="init_ctrl"),
        InitCode(targets=["target_patch"], initial_state=PauliEigenState.Z_plus, tag="init_target"),
        TransversalCNOT(targets=["ctrl_patch", "target_patch"], tag="transversal_cnot"),
        ReadoutCode(targets=["ctrl_patch"], tag="ro_ctrl"),
        ReadoutCode(targets=["target_patch"], tag="ro_target"),
        FreeCode(code_varname="ctrl_patch"),
        FreeCode(code_varname="target_patch"),
    ]

    compiler = QECCompiler(config={"objective_distance": 3, "primitives": PRIMITIVES_CONFIG})
    compiled_program = compiler.compile(program, show_progress=False)

    emitted_tags = [ob.tag for ob in compiled_program.observables]
    readout_tags = [tag for tag in emitted_tags if tag.startswith("readout_")]
    control_candidates = [tag for tag in readout_tags if "ctrl" in tag.lower()]
    target_candidates = [tag for tag in readout_tags if "target" in tag.lower()]
    if len(control_candidates) != 1 or len(target_candidates) != 1:
        raise ValueError(
            f"Could not uniquely identify readout tags: ctrl={control_candidates}, target={target_candidates}"
        )
    stim_observables = [[control_candidates[0]], [target_candidates[0]]]
    return compiled_program, stim_observables


compiled_program, stim_observables = compile_steane_transversal_cnot_test()

for p in compiled_program.to_stim_program(stim_observables).splitlines():
    print(p)

# Check graphlike distance via Stim DEM
probe_noise = StimLikeNoiseModel.from_stim_like_probabilities(
    after_clifford_depolarization=1e-3,
    after_reset_flip_probability=1e-3,
    before_measure_flip_probability=1e-3,
    before_round_data_depolarization=1e-3,
)
probe_circuit = stim.Circuit(compiled_program.to_stim_program(stim_observables, noise_model=probe_noise))
dem = probe_circuit.detector_error_model(allow_gauge_detectors=True, ignore_decomposition_failures=True)
graphlike_distance = len(dem.shortest_graphlike_error())
print(f"Graphlike distance: {graphlike_distance}")

RZ 1 0 4 3 6 2 5
RZ 18 17 14 15 19 16
REPEAT 3 {
    H 18 17 14
    TICK
    CX 17 2 18 4 14 3
    TICK
    CX 17 1 18 3 14 4
    TICK
    CX 17 5 18 2 14 1
    TICK
    CX 18 0 17 4 14 6
    TICK
    H 18 17 14
    TICK
    CX 1 16 0 15 3 19
    TICK
    CX 2 16 4 15 1 19
    TICK
    CX 3 15 4 19 5 16
    TICK
    CX 2 15 4 16 6 19
    TICK
    MRZ 18 17 14 15 19 16
}
RZ 11 12 7 10 9 13 8
RZ 18 17 14 15 19 16
REPEAT 3 {
    H 18 17 16
    TICK
    CX 16 7 18 10 17 8
    TICK
    CX 16 12 17 10 18 7
    TICK
    CX 18 9 16 8 17 7
    TICK
    CX 17 13 18 11
    TICK
    CX 16 11
    TICK
    H 18 17 16
    TICK
    CX 7 19 12 15 9 14
    TICK
    CX 7 15 10 19 11 14
    TICK
    CX 8 19 7 14 11 15
    TICK
    CX 8 15 13 19 10 14
    TICK
    MRZ 18 17 14 15 19 16
}
CNOT 5 12 0 13 6 9 2 8 3 10 4 7 1 11
RZ 23 21 17 18 14 15 25 24 20 16 19 22
REPEAT 3 {
    H 23 21 18 15 25 22
    TICK
    CX 15 5 23 7 21 0 22 12 18 8 25 1
    TICK
    CX 18 7 23 11 21 2 22 8 25 4 15 1
    TICK
    CX 2

In [8]:
noise_values = np.logspace(-5, -1, 9)
shots = 100

tasks = []
for physical_noise in noise_values:
    noise_model = StimLikeNoiseModel.from_stim_like_probabilities(
        after_clifford_depolarization=float(physical_noise),
        after_reset_flip_probability=float(physical_noise),
        before_measure_flip_probability=float(physical_noise),
        before_round_data_depolarization=float(physical_noise),
    )
    stim_program = compiled_program.to_stim_program(stim_observables, noise_model=noise_model)
    tasks.append(
        sinter.Task(
            circuit=stim.Circuit(stim_program),
            json_metadata={"d": 3, "p": float(physical_noise), "code": "3x_steane"},
        )
    )

print("Starting BPOSD benchmark run")
print(f"  shots={shots}, noise_points={len(noise_values)}, decoder=bposd")


def log_progress(progress: sinter.Progress) -> None:
    if progress.status_message:
        print(f"[bposd] {progress.status_message}")


collected_stats = sinter.collect(
    num_workers=4,
    tasks=tasks,
    decoders=["bposd"],
    custom_decoders={
        "bposd": SinterBpOsdDecoder(
            max_iter=0,
            bp_method="ms",
            ms_scaling_factor=0.625,
            osd_method="osd0",
            osd_order=0,
        )
    },
    max_shots=shots,
    progress_callback=log_progress,
    print_progress=True,
)

stats_by_noise = {float(stats.json_metadata["p"]): stats for stats in collected_stats}
logical_error_rates = np.array(
    [
        stats_by_noise[float(physical_noise)].errors / stats_by_noise[float(physical_noise)].shots
        for physical_noise in noise_values
    ]
)

result_title = "EPIC 3x Steane CNOT logical error rate"
result_description = (
    "Logical failure probability for ancilla-mediated CNOT on 3x [[7,1,3]] Steane codes "
    "(one for control, one for target, one as ancilla), using HM-based ZZ/XX product measurements "
    "with feedforward observable definition. A shot is counted as a logical fail when any tracked observable is 1. "
    "Noise uses EPIC StimLikeNoiseModel with matched p for reset flip, Clifford depolarization, "
    "pre-measurement flip, and round-data depolarization channels."
)

results_path = logical_rates_dir / "steane_cnot_rates.npz"
np.savez_compressed(
    results_path,
    distance=np.array([3], dtype=np.int64),
    noise_values=noise_values,
    logical_error_rates=logical_error_rates,
    shots=np.array([shots], dtype=np.int64),
    code=np.array(["3x_steane"]),
    title=np.array([result_title]),
    description=np.array([result_description]),
)

print(f"\nSaved 3x Steane CNOT rates to: {results_path.name}")
for physical_noise, logical_error_rate in zip(noise_values, logical_error_rates):
    print(f"  p={physical_noise:.2e} -> P_L={logical_error_rate:.6f}")

Starting 4 workers...


Starting BPOSD benchmark run
  shots=100, noise_points=9, decoder=bposd
[bposd] Starting 4 workers...


9 tasks left:
  workers decoder eta shots_left errors_seen json_metadata                              
        1   bposd   ?        100           0 d=3,p=1e-05,code=3x_steane                 
        1   bposd   ?        100           0 d=3,p=3.1622776601683795e-05,code=3x_steane
        1   bposd <1m         99           0 d=3,p=0.0001,code=3x_steane                
        1   bposd   ?        100           0 d=3,p=0.00031622776601683794,code=3x_steane
        0   bposd ?·∞        100           0 d=3,p=0.001,code=3x_steane                 
        0   bposd ?·∞        100           0 d=3,p=0.0031622776601683794,code=3x_steane 
        0   bposd ?·∞        100           0 d=3,p=0.01,code=3x_steane                  
        0   bposd ?·∞        100           0 d=3,p=0.03162277660168379,code=3x_steane   
        0   bposd ?·∞        100           0 d=3,p=0.1,code=3x_steane                   
6 tasks left:
  workers decoder eta shots_left errors_seen json_metadata                        

[bposd] 9 tasks left:
  workers decoder eta shots_left errors_seen json_metadata                              
        1   bposd   ?        100           0 d=3,p=1e-05,code=3x_steane                 
        1   bposd   ?        100           0 d=3,p=3.1622776601683795e-05,code=3x_steane
        1   bposd <1m         99           0 d=3,p=0.0001,code=3x_steane                
        1   bposd   ?        100           0 d=3,p=0.00031622776601683794,code=3x_steane
        0   bposd ?·∞        100           0 d=3,p=0.001,code=3x_steane                 
        0   bposd ?·∞        100           0 d=3,p=0.0031622776601683794,code=3x_steane 
        0   bposd ?·∞        100           0 d=3,p=0.01,code=3x_steane                  
        0   bposd ?·∞        100           0 d=3,p=0.03162277660168379,code=3x_steane   
        0   bposd ?·∞        100           0 d=3,p=0.1,code=3x_steane                   
[bposd] 9 tasks left:
  workers decoder eta shots_left errors_seen json_metadata        

KeyboardInterrupt


KeyboardInterrupt: 